# 01 - Silver cleaning

Reads bronze, applies all the cleaning steps from `utils/cleaning.py` and `utils/id_generation.py`,
and writes the result as a one-big-table to `marathos.silver.results_obt`.

The actual logic lives in utils so this notebook stays readable - it's mostly a pipeline of function calls.

*Some parts and solutions of this code was found through help from Claude*

In [0]:
import sys
import os

# Make repo root importable so we can use utils/
sys.path.append(os.path.abspath("../.."))

from pyspark.sql import functions as F

from utils.constants import BRONZE_RESULTS_TABLE, SILVER_RESULTS_OBT, SILVER_DIM_COUNTRY
from utils.io_helpers import read_table, write_df_to_table
from utils.cleaning import (
    split_event_distance,
    split_performance,
    add_event_type,
    add_validity_flag,
    clean_rows,
    normalize_country_codes,
    select_final_columns,
)
from utils.id_generation import add_event_id, add_athlete_id

## Read bronze and apply the cleaning pipeline

Each step transforms the dataframe a bit. Reading top to bottom shows the whole flow.

In [0]:
bronze_df = read_table(spark, BRONZE_RESULTS_TABLE)

silver_df = (
    bronze_df
    .transform(split_event_distance)
    .transform(split_performance)
    .transform(add_event_type)
    .transform(add_validity_flag)
    .transform(clean_rows)
    .transform(normalize_country_codes)
    .transform(add_event_id)
    .transform(add_athlete_id)
    .transform(select_final_columns)
)

## Enrich with country data (Bonus 1)

Left join the country dimension so every result row gets `country_name` and `continent`.
Left join keeps rows even if the country code doesn't have a match - we'd rather keep
the result and have a NULL country_name than drop the row.

In [0]:
country_df = read_table(spark, SILVER_DIM_COUNTRY)

silver_df = silver_df.join(
    country_df,
    silver_df["athlete_country"] == country_df["country_code"],
    how="left"
).drop("country_code")

## Verify before writing

Force Spark to run the whole pipeline so we can inspect the result
without committing to a table write.

In [0]:
# Force Spark to actually run the pipeline so we can inspect the result
# before writing it to a table
print(f"Rows: {silver_df.count():,}")
silver_df.printSchema()
silver_df.show(5, truncate=False)

## Write the OBT to silver

In [0]:
write_df_to_table(silver_df, SILVER_RESULTS_OBT)
print(f"Wrote silver OBT to {SILVER_RESULTS_OBT}")